# 02 - ASR (自动语音识别) 实验

本 notebook 用于演示和实验 `faster-whisper` ASR 模型的功能，特别是在流式处理场景下的应用。
主要内容包括：
1. 加载预处理后的音频数据。
2. 初始化 `FasterWhisperStreamer`。
3. 对单个音频块进行转录。
4. 模拟流式输入并进行转录，演示文本合并逻辑。
5. 实验不同的分段策略（VAD vs 固定块 - 虽然这更多是在 `StreamingPipeline` 中体现，这里可以简单演示其影响）。
6. 实验上下文窗口对转录结果的影响。
7. （可选）如果提供参考文本，计算词错误率 (WER)。

## 1. 初始化与配置

In [ ]:
import os
import sys
import glob
import time
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import jiwer # 用于 WER 计算
import matplotlib.pyplot as plt
import IPython.display as ipd

# 将项目根目录添加到 sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import (
    PROCESSED_AUDIO_DIR, TRANSCRIPTS_DIR, 
    ASR_MODEL_NAME, DEVICE,
    STREAMING_ASR_CHUNK_SECONDS, STREAMING_ASR_OVERLAP_SECONDS,
    VAD_SILENCE_MS, MIN_SPEECH_DURATION_S,
    TARGET_SAMPLE_RATE
)
from src.utils.audio_utils import AudioUtils
from src.asr.faster_whisper_streamer import FasterWhisperStreamer
from src.asr.audio_segmenter import AudioSegmenter
from src.utils.logging_utils import get_logger

logger = get_logger('ASRExperimentsNotebook')

In [ ]:
# 检查必要的目录
if not os.path.exists(PROCESSED_AUDIO_DIR):
    logger.warning(f"处理后音频目录 {PROCESSED_AUDIO_DIR} 不存在。请先运行数据预处理 notebook。")
    # os.makedirs(PROCESSED_AUDIO_DIR, exist_ok=True) # 或者在这里创建，但最好是预处理后就有

logger.info(f"ASR 模型: {ASR_MODEL_NAME}")
logger.info(f"运行设备: {DEVICE}")

## 2. 加载预处理的音频文件

我们将从 `data/processed_audio/` 目录加载一个示例音频文件。确保您已经运行了 `01_data_preprocessing.ipynb` 并生成了处理后的音频。

In [ ]:
processed_audio_files = glob.glob(os.path.join(PROCESSED_AUDIO_DIR, "*_processed.wav"))

if not processed_audio_files:
    logger.warning(f"在 {PROCESSED_AUDIO_DIR} 中没有找到 \"*_processed.wav\" 文件。")
    logger.warning("请先运行 01_data_preprocessing.ipynb 生成预处理音频。")
    # 为了能继续运行，创建一个虚拟的已处理音频（如果不存在）
    dummy_processed_path = os.path.join(PROCESSED_AUDIO_DIR, "dummy_audio_processed.wav")
    if not os.path.exists(dummy_processed_path):
        os.makedirs(PROCESSED_AUDIO_DIR, exist_ok=True) # 确保目录存在
        # 创建一个短的静音或简单音调的16kHz单声道音频
        sr_target = 16000
        duration_s = 3
        dummy_audio = np.zeros(int(sr_target * duration_s), dtype=np.float32) 
        # dummy_audio = 0.1 * np.sin(2 * np.pi * 220 * np.linspace(0, duration_s, int(sr_target*duration_s)) )
        sf.write(dummy_processed_path, dummy_audio, sr_target, subtype='FLOAT')
        logger.info(f"创建了虚拟的已处理音频文件: {dummy_processed_path}")
        processed_audio_files = [dummy_processed_path]

example_audio_path = None
if processed_audio_files:
    example_audio_path = processed_audio_files[0]
    logger.info(f"将使用以下音频文件进行实验: {example_audio_path}")
    
    audio_data, sr = AudioUtils.load_audio(example_audio_path, sample_rate=TARGET_SAMPLE_RATE)
    logger.info(f"加载音频: {os.path.basename(example_audio_path)}, 采样率: {sr} Hz, 时长: {len(audio_data)/sr:.2f}s")
    ipd.display(ipd.Audio(data=audio_data, rate=sr))
else:
    logger.error("无法找到或创建示例处理音频，请检查数据预处理步骤。")

## 3. 初始化 FasterWhisperStreamer

In [ ]:
try:
    asr_streamer = FasterWhisperStreamer(
        model_size=ASR_MODEL_NAME, 
        device=DEVICE,
        # compute_type="float16" # 可以根据GPU调整
    )
    logger.info(f"FasterWhisperStreamer 初始化成功。模型: {asr_streamer.model_size}, 设备: {asr_streamer.device}")
except Exception as e:
    logger.error(f"初始化 FasterWhisperStreamer 失败: {e}")
    asr_streamer = None # 确保后续代码可以检查

## 4. 对单个音频块进行转录 (非流式，用于基准测试或简单场景)

In [ ]:
if asr_streamer and 'audio_data' in locals():
    logger.info("测试对整个加载的音频进行一次性转录 (非流式):")
    start_time = time.time()
    # transcribe_audio_chunk 返回 (segments, info, transcription_time_s)
    segments, info, tt_s = asr_streamer.transcribe_audio_chunk(audio_data)
    end_time = time.time()
    
    full_transcript = "".join([s.text for s in segments])
    logger.info(f"  检测到语言: {info.language} (概率: {info.language_probability:.2f})")
    logger.info(f"  转录耗时 (模型报告): {tt_s:.3f} 秒")
    logger.info(f"  转录耗时 (端到端): {end_time - start_time:.3f} 秒")
    logger.info(f"  转录结果 (完整): '{full_transcript}'")

    # 显示带时间戳的词语
    logger.info("  词语和时间戳 (前5个片段):")
    for i, segment in enumerate(segments[:5]):
        logger.info(f"    片段 {i}: [{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")
        # for word in segment.words[:3]: # 显示每个片段的前3个词
        #     logger.info(f"      字: '{word.word}' [{word.start:.2f}s -> {word.end:.2f}s] (p={word.probability:.2f})")
else:
    logger.warning("ASR Streamer 未初始化或音频数据未加载，跳过单块转录测试。")

## 5. 模拟流式输入和转录

这里我们将模拟音频流，使用 `AudioSegmenter` 进行分块（固定长度），并逐步将这些块提供给 `FasterWhisperStreamer`。
我们将演示：
- 如何使用上下文窗口（`asr_context_window_chunks`）。
- 如何合并重叠部分的转录结果（基于词时间戳）。

注意：这部分逻辑与 `StreamingPipeline` 中的 `_process_audio_chunk_for_asr` 方法类似，但这里更侧重于逐步演示。

In [ ]:
if asr_streamer and 'audio_data' in locals():
    logger.info("\n测试模拟流式转录：")
    
    # --- 配置参数 ---
    # 使用 config.py 中的流式参数或在此处定义
    chunk_duration_s = STREAMING_ASR_CHUNK_SECONDS  # 例如 5秒一个块
    overlap_duration_s = STREAMING_ASR_OVERLAP_SECONDS # 例如 1秒重叠
    context_window_chunks = 3 # 使用3个块作为上下文
    min_chunk_size_s = 1.0 # 最小有效块长度

    segmenter = AudioSegmenter(
        use_vad=False, # 使用固定块演示流式
        chunk_duration_s=chunk_duration_s,
        overlap_duration_s=overlap_duration_s
    )

    # 使用 segmenter 获取固定长度的重叠块
    # segment_by_fixed_chunks 返回 (numpy_chunk, start_time_s, end_time_s) 的迭代器
    audio_chunks_with_time = list(segmenter.segment_by_fixed_chunks(audio_data, sr))
    
    if not audio_chunks_with_time:
        logger.warning("AudioSegmenter未能从示例音频中生成任何块。")
    else:
        logger.info(f"音频被分割为 {len(audio_chunks_with_time)} 个固定块 (时长: {chunk_duration_s}s, 重叠: {overlap_duration_s}s)")
        
        current_full_transcript_words = [] # [(word, start_abs, end_abs, probability)]
        processed_chunk_indices = [] # 用于跟踪哪些块已被合并
        
        for i in range(len(audio_chunks_with_time)):
            # 1. 构建当前ASR的输入块 (带上下文)
            # 上下文是当前块之前的 context_window_chunks - 1 个块
            start_idx = max(0, i - context_window_chunks + 1)
            end_idx = i + 1 # 当前块包括在内
            
            context_chunks_info = audio_chunks_with_time[start_idx:end_idx]
            
            if not context_chunks_info:
                continue

            # 拼接上下文块的音频数据
            # 需要正确处理重叠，或者简单拼接然后让ASR处理
            # FasterWhisperStreamer.transcribe_audio_chunk 期望单个 numpy 数组
            # 这里的拼接方式是为了模拟将多个块送入ASR进行一次识别的场景
            # 我们需要的是针对当前块(i)及其左侧上下文的组合音频
            current_asr_input_np = np.concatenate([c[0] for c in context_chunks_info])
            input_chunk_abs_start_time_s = context_chunks_info[0][1] # 第一个块的绝对开始时间

            # 块太短可能导致Whisper表现不佳或出错
            if len(current_asr_input_np) / sr < min_chunk_size_s:
                logger.info(f"  步骤 {i+1}/{len(audio_chunks_with_time)}: 输入块太短 ({len(current_asr_input_np)/sr:.2f}s)，跳过。")
                continue
            
            logger.info(f"\n  步骤 {i+1}/{len(audio_chunks_with_time)}: 转录块 {start_idx}-{i} (共{len(context_chunks_info)}块, 音频时长 {len(current_asr_input_np)/sr:.2f}s)")

            # 2. 调用ASR进行转录
            segments, info, tt_s = asr_streamer.transcribe_audio_chunk(current_asr_input_np) # word_timestamps=True 默认开启
            logger.info(f"    ASR耗时: {tt_s:.3f}s, 语言: {info.language}")
            
            newly_transcribed_words_abs = []
            for segment in segments:
                for word in segment.words:
                    # 将相对时间戳转换为绝对时间戳 (相对于整个音频流的开始)
                    abs_start = input_chunk_abs_start_time_s + word.start
                    abs_end = input_chunk_abs_start_time_s + word.end
                    newly_transcribed_words_abs.append((word.word, abs_start, abs_end, word.probability))
            
            if not newly_transcribed_words_abs:
                logger.info("    此步骤无新词转录。")
                continue

            # 3. 文本合并逻辑 (简化版，参考 StreamingPipeline 中的实现)
            # 我们只合并当前处理的块 (i) 中不与前序块重叠的部分，或者基于时间戳替换
            # 这个示例主要关注如何从上下文窗口中提取当前块的最新信息

            # 目标是获取块 `i` (audio_chunks_with_time[i]) 的转录结果
            # 这个块在 `current_asr_input_np` 中的相对开始时间是 context_chunks_info 中最后一个块的开始时间 (相对于 current_asr_input_np)
            current_target_chunk_info = audio_chunks_with_time[i] # (audio_np, abs_start_s, abs_end_s)
            target_chunk_abs_start_s = current_target_chunk_info[1]
            target_chunk_abs_end_s = current_target_chunk_info[2]

            final_words_for_this_step = []
            # 从 newly_transcribed_words_abs 中筛选出属于当前目标块 target_chunk_abs_start_s 之后的部分
            # 并且如果这是最后一个块，则全部采纳；否则，只采纳到下一个块开始之前的部分（即不包含overlap到下一个块的部分）
            effective_end_s = target_chunk_abs_end_s
            if i < len(audio_chunks_with_time) - 1: # 如果不是最后一个块
                # 只取到当前块的非重叠部分末尾，即下一个块的开始时间
                # 如果块之间有间隙，则取当前块的结束；如果重叠，取当前块长度减去重叠部分
                # 简单处理：取当前块的实际结束时间 target_chunk_abs_end_s，但在合并时会处理
                # 更精确的逻辑是只取 target_chunk_abs_start_s 到 (target_chunk_abs_start_s + chunk_duration_s - overlap_duration_s)
                # 或者，一个更鲁棒的方法是总是替换现有时间范围内的词
                pass # 复杂的合并在 pipeline 中，这里简化
            
            # 移除在 `current_full_transcript_words` 中时间戳从 `target_chunk_abs_start_s` 开始的词语
            # 因为这些将被新转录的词语所替代 (由于上下文的存在，新转录可能更准确)
            temp_merged_words = []
            # 保留所有在新片段开始之前的词语
            for w_obj in current_full_transcript_words:
                w_text, w_start, w_end, w_prob = w_obj
                if w_end <= target_chunk_abs_start_s: # 保留完全在此次处理范围之前的
                    temp_merged_words.append(w_obj)
                # 也可以有更细致的逻辑处理部分重叠的词
            
            current_full_transcript_words = temp_merged_words
            
            # 添加来自当前目标块的新词语 (或更晚的词语，如果ASR结果包含了超出当前块的预测)
            added_new_word_this_step = False
            for word, w_start_abs, w_end_abs, w_prob in newly_transcribed_words_abs:
                # 只添加那些在中目标块开始时间之后的词语
                # 并且，如果不是最后一个原始块，则要小心不要添加太多来自未来重叠部分的内容
                # 这里的简化逻辑：如果一个词的开始时间 >= target_chunk_abs_start_s 就认为它是新的、有效的
                # 并假设新转录的上下文结果总是更优的
                if w_start_abs >= target_chunk_abs_start_s: 
                    # 避免重复添加 (基于时间和词)
                    is_duplicate = False
                    for _, prev_s, prev_e, _ in current_full_transcript_words:
                        if abs(prev_s - w_start_abs) < 0.1 and abs(prev_e - w_end_abs) < 0.1: # 时间上非常接近
                           # 更严格的检查可能需要看词本身，但这里仅基于时间
                           pass # 可能需要更复杂的去重
                    if not is_duplicate:
                        current_full_transcript_words.append((word, w_start_abs, w_end_abs, w_prob))
                        added_new_word_this_step = True
            
            if added_new_word_this_step:
                # 对 current_full_transcript_words 按开始时间排序
                current_full_transcript_words.sort(key=lambda x: x[1])
                
                # 去重相邻相同的词 (简单版本)
                deduplicated_words = []
                if current_full_transcript_words:
                    deduplicated_words.append(current_full_transcript_words[0])
                    for k in range(1, len(current_full_transcript_words)):
                        # 如果当前词与上一个词文本不同，或者开始时间有显著差异，则添加
                        if current_full_transcript_words[k][0] != current_full_transcript_words[k-1][0] or \\
                           abs(current_full_transcript_words[k][1] - current_full_transcript_words[k-1][1]) > 0.01: # 时间差大于10ms
                            deduplicated_words.append(current_full_transcript_words[k])
                current_full_transcript_words = deduplicated_words

            final_text_output = "".join([w[0] for w in current_full_transcript_words])
            logger.info(f"    累计转录 ({len(current_full_transcript_words)} words): {final_text_output[:200]}...")

        logger.info("\n模拟流式转录完成。")
        final_streamed_transcript = "".join([word[0] for word in current_full_transcript_words])
        logger.info(f"最终流式转录结果: {final_streamed_transcript}")

else:
    logger.warning("ASR Streamer 未初始化或音频数据未加载，跳过流式转录测试。")

## 6. （可选）与参考文本比较并计算 WER

如果 `data/transcripts/` 目录下有对应的参考文本文件，我们可以计算 WER。

In [ ]:
if example_audio_path and os.path.exists(TRANSCRIPTS_DIR):
    base_name = os.path.splitext(os.path.basename(example_audio_path))[0]
    # 假设参考文件名与处理后的音频文件名（去掉_processed后缀）对应
    if base_name.endswith("_processed"):
        ref_base_name = base_name[:-len("_processed")]
    else:
        ref_base_name = base_name
    
    reference_transcript_path = os.path.join(TRANSCRIPTS_DIR, f"{ref_base_name}.txt")

    if os.path.exists(reference_transcript_path):
        with open(reference_transcript_path, 'r', encoding='utf-8') as f:
            reference_text = f.read().strip()
        logger.info(f"加载参考文本: {reference_transcript_path}")
        logger.info(f"  参考内容: '{reference_text[:100]}...'" )

        if 'full_transcript' in locals() and full_transcript:
            wer_full = jiwer.wer(reference_text, full_transcript)
            logger.info(f"  整段转录 WER: {wer_full:.4f}")
        else:
            logger.warning("  未找到 'full_transcript' (整段转录结果) 用于计算WER。")
            
        if 'final_streamed_transcript' in locals() and final_streamed_transcript:
            wer_streamed = jiwer.wer(reference_text, final_streamed_transcript)
            logger.info(f"  流式转录 WER: {wer_streamed:.4f}")
        else:
            logger.warning("  未找到 'final_streamed_transcript' (流式转录结果) 用于计算WER。")

    else:
        logger.warning(f"未找到参考文本文件: {reference_transcript_path}，跳过WER计算。")
else:
    logger.warning("示例音频路径或文本记录目录未定义，跳过WER计算。")

## 7. 结论与进一步实验方向

这个 notebook 演示了 `FasterWhisperStreamer` 的基本用法和流式处理的模拟。

可以进一步的实验包括：
- **不同 `compute_type` 的性能**: 如 `float16`, `int8_float16`, `int8` (如果硬件支持)。
- **VAD 参数调优**: 调整 `AudioSegmenter` 中 VAD 的参数，观察其对流式结果的影响。 (这更多是在 `StreamingPipeline` 中集成测试)
- **更长的音频文件**: 测试在更长的对话或音频上的表现。
- **不同 `beam_size`**: `faster-whisper` 支持beam search, 调整 `beam_size` 参数看对准确率和延迟的影响。
- **多语言和翻译**: 如果需要，测试 `language` 参数和 `task="translate"`。
- **更精细的文本合并策略**: `StreamingPipeline` 中的合并逻辑比本 notebook 中的演示更复杂，可以深入研究和优化。

In [ ]:
logger.info("ASR 实验 Notebook 执行完毕。")

---